In [2]:
import json
import time
from itertools import product

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [12]:
# data source 1: ETER university funding and outcomes
eu_country_codes = [
    'AT','BE','BG','HR','CY','CZ','DK','EE','FI','FR',
    'DE','GR','HU','IE','IT','LV','LT','LU','MT','NL',
    'PL','PT','RO','SK','SI','ES','SE'
]

eter_raw = pd.read_csv('../datafiles/university_funding.csv')

df = eter_raw[
    eter_raw['Country_Code'].isin(eu_country_codes) &
    (eter_raw['Reference_year'] == 2020)
].reset_index(drop=True)
print(f'ETER EU 2020 shape: {df.shape}')

funding_cols = [
    'Total_Current_revenues_(EURO)',
    'Basic_government_allocation_(EURO)',
    'Student_fees_funding_(EURO)',
    'Total_third_party_funding_(EURO)',
    'Personnel_expenditure_(EURO)',
    'R&D_Expenditure_(EURO)'
]
result_cols = [
    'Total_students_enrolled_ISCED_5_7',
    'Total_graduates_ISCED_5_7',
    'Total_graduates_at_ISCED_8'
]
keeping_cols = [
    'ETER_ID', 'English_Institution_Name', 'Institution_Name',
    'Reference_year', 'Country_Code', 'Name_of_the_city'
] + funding_cols + result_cols

df = df[keeping_cols].copy()

for col in funding_cols + result_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.reset_index(drop=True, inplace=True)
print(df.shape)
print(df[funding_cols + result_cols].describe().to_string())

ETER EU 2020 shape: (2033, 39)
(2033, 15)
       Total_Current_revenues_(EURO)  Basic_government_allocation_(EURO)  Student_fees_funding_(EURO)  Total_third_party_funding_(EURO)  Personnel_expenditure_(EURO)  R&D_Expenditure_(EURO)  Total_students_enrolled_ISCED_5_7  Total_graduates_ISCED_5_7  Total_graduates_at_ISCED_8
count                   8.270000e+02                        3.510000e+02                 7.350000e+02                      7.920000e+02                  7.880000e+02            3.170000e+02                        1742.000000                1746.000000                  847.000000
mean                    1.088374e+08                        9.765850e+07                 7.682322e+06                      2.231926e+07                  7.488863e+07            1.460837e+07                        8310.347325                1753.530928                  100.517119
std                     1.723579e+08                        1.233880e+08                 2.085014e+07                 

In [13]:
# data source 2: QS world university rankings
eu_country_names = [
    'Austria','Belgium','Bulgaria','Croatia','Cyprus','Czech Republic',
    'Denmark','Estonia','Finland','France','Germany','Greece','Hungary',
    'Ireland','Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands',
    'Poland','Portugal','Romania','Slovakia','Slovenia','Spain','Sweden'
]

qs_raw = pd.read_excel('../datafiles/university_world_rankings.xlsx', header=1)
qs_raw.columns = qs_raw.iloc[0]
qs_raw = qs_raw.drop(0).reset_index(drop=True)

df_2 = qs_raw[qs_raw['Country/Territory'].isin(eu_country_names)].reset_index(drop=True)
df_2.to_csv('../datafiles/university_world_rankings.csv', index=False)
print(f'QS EU shape: {df_2.shape}')

qs_score_cols = ['Overall SCORE', 'AR SCORE', 'ER SCORE', 'FSR SCORE',
                 'CPF SCORE', 'IFR SCORE', 'ISR SCORE', 'EO SCORE', 'SUS SCORE']
for col in qs_score_cols:
    df_2[col] = pd.to_numeric(df_2[col], errors='coerce')

print(df_2[['Name', 'Country/Territory'] + qs_score_cols].head(10).to_string())

QS EU shape: (311, 31)
0                                               Name Country/Territory  Overall SCORE  AR SCORE  ER SCORE  FSR SCORE  CPF SCORE  IFR SCORE  ISR SCORE  EO SCORE  SUS SCORE
0                     Technical University of Munich           Germany           90.2      92.0      99.7       70.5       92.6       86.3       98.9      57.3       87.8
1                                     PSL University            France           88.6      82.6      98.5       98.6       85.1       69.6       75.5      97.5       87.8
2                    Institut Polytechnique de Paris            France           85.4      58.1      99.8       94.8       97.3       99.9       99.2      99.1       77.9
3                     Delft University of Technology       Netherlands           84.3      86.2      90.4       49.1       84.6      100.0       94.8      66.0       96.3
4                        The University of Amsterdam       Netherlands           81.5      92.5      72.6       27.5      

In [19]:
# fuzzy matching: aka trying to see if there are missed matches between the two datasets, to maximize the number of universities we can analyze
import difflib

country_map = dict(zip(eu_country_names, eu_country_codes))

# find unmatched QS universities
matched_qs_names = set(merge1['Name'].tolist() + merge2['Name'].tolist())
unmatched_qs = df_2[~df_2['Name'].isin(matched_qs_names)].reset_index(drop=True)

results = []
for _, row in unmatched_qs.iterrows():
    qs_name = row['Name']
    country_code = country_map.get(row['Country/Territory'])
    if not country_code:
        continue
    eter_country = df[df['Country_Code'] == country_code]
    candidates_en = eter_country['English_Institution_Name'].dropna().tolist()
    candidates_local = eter_country['Institution_Name'].dropna().tolist()
    all_candidates = list(set(candidates_en + candidates_local))
    if not all_candidates:
        continue
    matches = difflib.get_close_matches(qs_name, all_candidates, n=1, cutoff=0.0)
    if matches:
        match = matches[0]
        score = round(difflib.SequenceMatcher(None, qs_name.lower(), match.lower()).ratio() * 100, 1)
    else:
        match, score = '', 0
    results.append({
        'QS_name': qs_name,
        'country': row['Country/Territory'],
        'proposed_ETER_name': match,
        'similarity': score,
        'accept': ''
    })

results_df = pd.DataFrame(results).sort_values('similarity', ascending=False).reset_index(drop=True)
results_df.to_csv('../datafiles/proposed_matches.csv', index=False)

print(f'Proposed matches saved to proposed_matches.csv')
print(f'Total to review: {len(results_df)}')
print(f'High confidence (>=85): {len(results_df[results_df["similarity"] >= 85])}')
print(f'Medium (70-84): {len(results_df[(results_df["similarity"] >= 70) & (results_df["similarity"] < 85)])}')
print(f'Low (<70): {len(results_df[results_df["similarity"] < 70])}')
print()
print(results_df[['QS_name', 'proposed_ETER_name', 'similarity', 'country']].to_string())

Proposed matches saved to proposed_matches.csv
Total to review: 121
High confidence (>=85): 77
Medium (70-84): 32
Low (<70): 12

                                                      QS_name                                            proposed_ETER_name  similarity      country
0                                     Université Paris-Saclay                                       Université Paris-saclay       100.0       France
1                                             Umeå University                                               Umeå university       100.0       Sweden
2                             Università degli Studi di Udine                               Università degli Studi di UDINE       100.0        Italy
3                             Università degli Studi Roma Tre                               Università degli Studi ROMA TRE       100.0        Italy
4                 Athens University of Economics And Business                   Athens University of Economics and Business   

In [1]:
qs_keep = ['Name'] + qs_score_cols + ['Rank', 'Size', 'Focus', 'Research', 'Status']
qs_merge = df_2[[c for c in qs_keep if c in df_2.columns]].copy()

# pass 1: english name
merge1 = pd.merge(df, qs_merge, left_on='English_Institution_Name', right_on='Name', how='inner')
print(f'Merge (English name): {len(merge1)} rows')

# pass 2: local name
unmatched = df[~df['English_Institution_Name'].isin(merge1['English_Institution_Name'])]
merge2 = pd.merge(unmatched, qs_merge, left_on='Institution_Name', right_on='Name', how='inner')
print(f'Merge (local name): {len(merge2)} rows')

# pass 3: accepted fuzzy matches
accepted = pd.read_csv('../datafiles/proposed_matches.csv')
accepted = accepted[accepted['accept'] == True]
print(f'Accepted fuzzy matches: {len(accepted)}')

name_map = dict(zip(accepted['proposed_ETER_name'], accepted['QS_name']))
already_matched = set(merge1['English_Institution_Name'].tolist() + merge2['English_Institution_Name'].tolist())

df['QS_match_name'] = df['English_Institution_Name'].map(name_map).fillna(
    df['Institution_Name'].map(name_map)
)
merge3 = pd.merge(
    df[df['QS_match_name'].notna() & ~df['English_Institution_Name'].isin(already_matched)],
    qs_merge,
    left_on='QS_match_name', right_on='Name',
    how='inner'
)
print(f'Merge (fuzzy accepted): {len(merge3)} rows')

# combine all three passes
merged = pd.concat([merge1, merge2, merge3], ignore_index=True)

merged['graduation_rate'] = (
    merged['Total_graduates_ISCED_5_7'] / merged['Total_students_enrolled_ISCED_5_7']
).clip(0, 1)

print(f'\nFinal merged shape: {merged.shape}')
print(merged[['English_Institution_Name', 'Country_Code',
              'Total_Current_revenues_(EURO)', 'Overall SCORE',
              'graduation_rate']].head(10).to_string())

merged.to_csv('../datafiles/budget_funding_plan.csv', index=False)
print('\nSaved to budget_funding_plan.csv')

NameError: name 'qs_score_cols' is not defined